# Tarea 6

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import yfinance as yf

from scipy.optimize import minimize

In [2]:
from dotenv import load_dotenv
from pathlib import Path
import os
import sys

load_dotenv()

PROJECT_ROOT = Path(os.environ["PORTFOLIO_ROOT"]).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [35]:
from src.asset_allocation import PortfolioElementaryMetrics

from src.asset_allocation import PortfolioOptimizationPostModern
from src.asset_allocation import PortfolioElementaryAnalysis

## A) Implementación de funciones

Punto a) Implementen tres funciones que optimicen portafolios utilizando: mínima semivarianza, ratio omega y semivarianza objetivo, empleando la función `scipy.optimize.minimize`. (40%)

In [5]:
# function aux
OPTIMIZATION_TOL = 1e-50

def portfolio_var(weight : np.ndarray, cov_matrix):
    return weight.T @ cov_matrix @ weight

def weight_sum(weight : np.ndarray):
    assets = len(weight)
    return weight @ np.ones(assets) - 1

def _asset_returns(prices : pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().dropna()

def _initial_weight(n_assets : int) -> np.ndarray:
    return np.ones(n_assets) / n_assets

def _weight_bounds(n_assets : int):
    return [(0, 1)] * n_assets

def _optimize_weights(objective, n_assets : int) -> np.ndarray:
    opt = minimize(
        fun=objective,
        x0=_initial_weight(n_assets),
        bounds=_weight_bounds(n_assets),
        constraints={"fun": weight_sum, "type": "eq"},
        tol=OPTIMIZATION_TOL
    )
    print(opt)
    return opt.x

def _build_semivariance_matrix(
    downside_risk : np.ndarray,
    corr : pd.DataFrame,
    n_assets : int,
):
    return downside_risk.reshape(n_assets, 1) @ downside_risk.reshape(1, n_assets) * corr



In [41]:
def min_semivar(prices : pd.DataFrame, umbral : float = 0) -> np.ndarray:
    returns = _asset_returns(prices)
    n_assets = len(returns.columns)
    corr = returns.corr()

    below_threshold = returns[returns < umbral].fillna(0)
    downside_risk = np.array(below_threshold.std())
    semivar_matrix = _build_semivariance_matrix(downside_risk, corr, n_assets)

    return _optimize_weights(
        objective=lambda weight: portfolio_var(weight, semivar_matrix),
        n_assets=n_assets,
    )


def ratio_omega(prices : pd.DataFrame, umbral : float = 0) -> np.ndarray:
    returns = _asset_returns(prices)
    n_assets = len(returns.columns)

    below_threshold = returns[returns < umbral].fillna(0)
    above_threshold = returns[returns > umbral].fillna(0)

    downside_risk = np.array(below_threshold.std())
    upside_risk = np.array(above_threshold.std())
    omega = upside_risk / downside_risk

    return _optimize_weights(
        objective=lambda weight: (-1) * (omega @ weight),
        n_assets=n_assets,
    )


def target_semivar(prices : pd.DataFrame, benchmark : pd.DataFrame = None, umbral : float = 0):
    returns_assets = _asset_returns(prices)
    n_assets = len(returns_assets.columns)
    corr = returns_assets.corr()

    if umbral != 0 and type(umbral) is float:
        diff = returns_assets - umbral
    elif benchmark is not None:
        returns_benchmark = _asset_returns(benchmark)
        diff = returns_assets - returns_benchmark.values
        print(diff)
    else:
        return None

    below_target = diff[diff < 0].fillna(0)
    target_downside = np.array(below_target.std())
    target_semivar_matrix = _build_semivariance_matrix(target_downside, corr, n_assets)

    return _optimize_weights(
        objective=lambda weight: portfolio_var(weight, target_semivar_matrix),
        n_assets=n_assets,
    )

## B) Construccion del portafolio

In [43]:
start_date = "2010-01-01"
end_date = "2026-01-01"

In [49]:
# Descarga de Datos
tickets = ['VALUEGFO.MX', 'FPLUS16.MX', 'KUOB.MX', 'LAMOSA.MX']

prices=yf.download(tickets, start=start_date, end=end_date, progress=False)['Close']
prices

/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_

Ticker,FPLUS16.MX,KUOB.MX,LAMOSA.MX,VALUEGFO.MX
Date,,,,
2010-01-04,NaN,8.443341,9.843429,9.022394
2010-01-05,NaN,8.443341,9.922174,9.312104
2010-01-06,NaN,8.663603,10.008798,9.312104
2010-01-07,NaN,8.810444,10.008798,9.312104
2010-01-08,NaN,8.443341,10.079669,9.312104
...,...,...,...,...
2025-12-24,5.15,57.990002,102.989998,73.500000
2025-12-26,5.15,57.990002,102.989998,73.500000
2025-12-29,5.57,60.000000,102.989998,73.500000


In [47]:
benchmark = ["^MXX"]

benchmark_price = yf.download(benchmark, start=start_date, end=end_date, progress=False)["Close"]

/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


### Minima semivarianza

In [52]:
result = min_semivar(prices)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 2.7467087050607902e-05
           x: [ 1.377e-01  3.483e-01  4.530e-01  6.098e-02]
         nit: 28
         jac: [ 5.493e-05  5.493e-05  5.493e-05  5.493e-05]
        nfev: 187
        njev: 28
 multipliers: [ 5.493e-05]


In [53]:
dict(zip(prices.columns, result))

{'FPLUS16.MX': np.float64(0.13774396030594852),
 'KUOB.MX': np.float64(0.348270043863875),
 'LAMOSA.MX': np.float64(0.4530015721234317),
 'VALUEGFO.MX': np.float64(0.06098442370674476)}

In [ ]:
weight = np.ones(len(tickets)) / len(tickets)

Portfolio1 = PortfolioOptimizationPostModern(
    tickers=tickets,
    start=start_date,
    end=end_date,
    weight=weight
)

OptimizationResult = Portfolio1.optimize_minimum_semivariance()

/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/home/rigodev/ITESO/06/portafolios/env/lib/python3.12/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_

PostModernOptimizationResult(objective='minimum_semivariance', success=True, status=0, message='Optimization terminated successfully', weights=array([0.0609866 , 0.13775705, 0.34817261, 0.45308373]), weights_by_ticker=VALUEGFO.MX    0.060987
FPLUS16.MX     0.137757
KUOB.MX        0.348173
LAMOSA.MX      0.453084
Name: weight, dtype: float64, expected_return=0.106211394680339, semivariance=0.006921706204959004, downside_risk=0.08319679203526423, omega=1.1824730003705908, objective_value=0.006921706204959004, iterations=15)

In [56]:
Portfolio1.portfolio_semivariance()

0.012264199390355418

In [60]:
weight_min_semivar = OptimizationResult.weights

np.sqrt(weight_min_semivar.T @ Portfolio1.semivariance_matrix() @ weight_min_semivar)

np.float64(0.08319679203526423)

In [62]:
np.sqrt(Portfolio1.portfolio_semivariance())

np.float64(0.11074384583513171)

### Ratio omega

In [10]:
result = ratio_omega(prices)

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: -1.3464046699859697
           x: [ 7.586e-16  1.292e-16  1.000e+00  0.000e+00]
         nit: 7
         jac: [-1.267e+00 -1.159e+00 -1.346e+00 -1.238e+00]
        nfev: 46
        njev: 7
 multipliers: [-1.346e+00]


In [11]:
dict(zip(prices.columns, result))

{'FPLUS16.MX': np.float64(7.585759808368111e-16),
 'KUOB.MX': np.float64(1.2917405516277203e-16),
 'LAMOSA.MX': np.float64(0.9999999999999991),
 'VALUEGFO.MX': np.float64(0.0)}